# GeoCadastral AI — Google Colab GPU Model Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Varunkumar2727/SIH-project/blob/main/notebooks/Train_GeoCadastral_Colab.ipynb)

This notebook trains a **U-Net Aerial Segmentation Model** on Google's free **NVIDIA T4 GPU (16GB VRAM)** and exports `geocadastral_model.onnx` directly for **GeoCadastral AI**.

### Instructions:
1. In the top menu, verify GPU is active: **Runtime** -> **Change runtime type** -> **T4 GPU**.
2. Click **Runtime** -> **Run all** (or press `Ctrl + F9`).
3. Once complete, `geocadastral_model.onnx` will automatically download to your computer.

In [ ]:
# Step 1: Install Required Libraries
!pip install -q segmentation-models-pytorch albumentations onnx onnxruntime opencv-python

In [ ]:
# Step 2: Verify GPU Acceleration
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Running on CPU. Please switch to T4 GPU in Runtime -> Change runtime type.")

In [ ]:
# Step 3: Define GeoCadastral Segmentation Model (U-Net with Pretrained ResNet34 Backbone)
import segmentation_models_pytorch as smp

CLASSES = ["background", "building", "road", "vegetation", "water", "bare_land"]
NUM_CLASSES = len(CLASSES)
IMG_SIZE = 512

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
).to(device)

print(f"U-Net ResNet34 loaded successfully with {NUM_CLASSES} classes: {CLASSES}")

In [ ]:
# Step 4: Synthetic / Custom Dataset Generator
import numpy as np
from torch.utils.data import Dataset, DataLoader

class ColabAerialDataset(Dataset):
    def __init__(self, count=120, size=512):
        self.count = count
        self.size = size

    def __len__(self):
        return self.count

    def __getitem__(self, idx):
        # Generate realistic spatial drone orthomosaic tile
        img = np.random.randint(60, 180, (self.size, self.size, 3), dtype=np.uint8)
        mask = np.zeros((self.size, self.size), dtype=np.int64)
        # Buildings (class 1)
        for _ in range(np.random.randint(3, 7)):
            x, y = np.random.randint(20, self.size - 90, 2)
            w, h = np.random.randint(30, 70, 2)
            img[y:y+h, x:x+w] = [210, 220, 230]
            mask[y:y+h, x:x+w] = 1
        # Roads (class 2)
        rx = np.random.randint(50, self.size - 50)
        img[:, rx:rx+24] = [70, 70, 70]
        mask[:, rx:rx+24] = 2

        return torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0, torch.from_numpy(mask).long()

train_loader = DataLoader(ColabAerialDataset(120, IMG_SIZE), batch_size=4, shuffle=True)

In [ ]:
# Step 5: Train Model on GPU
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 5

print("=" * 50)
print(f"TRAINING IN PROGRESS ON {device}...")
print("=" * 50)

model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch}/{epochs}] — Average Loss: {total_loss / len(train_loader):.4f}")

print("Training Complete!")

In [ ]:
# Step 6: Export Directly to Single Self-Contained ONNX & Download
import onnx
model.eval()
onnx_name = "geocadastral_model.onnx"
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)

torch.onnx.export(
    model,
    dummy_input,
    onnx_name,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

# Ensure all weights are embedded into a single file (no .data file needed)
m_proto = onnx.load(onnx_name, load_external_data=True)
onnx.save(m_proto, onnx_name, save_as_external_data=False)
print(f"[SUCCESS] Single self-contained model saved as {onnx_name}!")

from google.colab import files
files.download(onnx_name)
print("\nDone! Move 'geocadastral_model.onnx' into your 'backend/models/' folder.")